# Countdown: GRPO и reward hacking

Перед запуском выберите **Runtime → Change runtime type → T4 GPU**. Загрузите проект на GitHub и замените `REPO_URL` ниже. Сравниваются strict и naive награды при одинаковых Qwen3-0.6B, LoRA, seed и бюджете. Полный запуск может занять несколько часов.

In [ ]:
REPO_URL = 'https://github.com/YOUR_USERNAME/rl-session-assignment.git'  # @param {type:'string'}
from pathlib import Path
repo_name = REPO_URL.rstrip('/').split('/')[-1].replace('.git', '')
repo_root = Path('/content') / repo_name
if not repo_root.exists():
    !git clone $REPO_URL $repo_root
%cd $repo_root/track3_countdown_grpo

In [ ]:
%pip install -q -r requirements.txt
import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime first'
!python -m countdown_rl.data --config configs/full.json
!pytest -q

## Сначала короткая проверка, затем полный эксперимент
Smoke запускает 2 шага и 8 задач. Он проверяет только память и совместимость. Для отчёта установите `MODE='full'`: 2 условия × 3 seed × 300 шагов и 240 отложенных задач на каждом чекпоинте.

In [ ]:
MODE = 'smoke'  # @param ['smoke', 'full']
!python run.py --config configs/$MODE.json --output results/$MODE --outputs outputs/$MODE

In [ ]:
from IPython.display import display, Image, Markdown
results = Path('results') / MODE
display(Markdown((results / 'RESULTS.md').read_text(encoding='utf-8')))
for name in ['training_reward.png', 'heldout_accuracy.png', 'kl_divergence.png', 'response_length.png']:
    if (results / name).exists():
        display(Image(filename=str(results / name)))

In [ ]:
import shutil
archive = shutil.make_archive(f'/content/countdown_{MODE}_results', 'zip', results)
print('Archive:', archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass